# Following a recurrent state through a sequence

Suppose a review says “good then bad.” What changes if we read “bad then good”
instead? Both phrases contain exactly the same words. A count of the words
would give us the same result for both, but the order changes what happened
last. Let us see how a recurrent neural network (RNN) can represent that
difference.

We will use a tiny RNN with one number as its hidden state, the value it passes
from one step to the next. Then we will split a phrase in two and find out what
happens when we carry the state across the split. This notebook runs with
PyTorch on a CPU.

## Setting up a one-number state

We will give `good` the input value 1, `then` the value 0, and `bad` the
value −1. At each step the RNN adds the current input to half of the previous
state and applies the `tanh` activation. This keeps the state between −1 and 1.
In other words, the same word can produce a different state depending on what
the RNN has already read.

In the following code, `weight_ih_l0` multiplies the input and
`weight_hh_l0` multiplies the previous hidden state. We set both weights by
hand so we can follow the calculation. When we train a text classifier later,
the model will learn its own weights.

In [ ]:
import torch
from torch import nn

rnn = nn.RNN(input_size=1, hidden_size=1, bias=False, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.fill_(1.0)
    rnn.weight_hh_l0.fill_(0.5)

word_values = {"good": 1.0, "then": 0.0, "bad": -1.0}

def encode(phrase):
    values = [word_values[word] for word in phrase.split()]
    # One phrase per batch, one value per word.
    return torch.tensor(values).reshape(1, -1, 1)

## Reading the same words in a different order

For the first phrase, the state should start positive after `good`. When the
RNN reads `then`, the input is zero, so only the previous state can influence
the result. Predict what happens when the RNN reaches `bad`, and whether the
final state has the same sign for the reversed phrase.

When we call the RNN, it returns one state per word in `states` and also
returns the final state separately. This second value will be useful when we
read a phrase in two chunks.

In [ ]:
phrases = ("good then bad", "bad then good")

with torch.no_grad():
    for phrase in phrases:
        states, final_state = rnn(encode(phrase))
        trace = "  ".join(
            f"{word}: {state:+.3f}"
            for word, state in zip(phrase.split(), states.flatten().tolist())
        )
        print(f"{phrase:13} | {trace} | final: {final_state.item():+.3f}")

After reading `then`, we still have a trace of the first word, but the state's
magnitude has fallen. In this example the last word is strong enough to change
the sign. Reversing the words therefore reverses the sign of the final state.
These values come from the weights we chose. They give us a way to see the
effect of order before we train an RNN on real reviews.

## Carrying the state across chunks

Now suppose the first phrase reaches us one part at a time. We will give the
RNN `good` first and `then bad` next. If we pass the state from the first
call into the second call, the RNN should behave as if the phrase arrived all
at once. If we leave out that state, PyTorch starts the second call with zeros.

In [ ]:
sequence = encode(phrases[0])
first_chunk, second_chunk = sequence[:, :1], sequence[:, 1:]

with torch.no_grad():
    full_steps, full_final = rnn(sequence)
    first_steps, carried_state = rnn(first_chunk)
    second_steps, carried_final = rnn(second_chunk, carried_state)
    _, reset_final = rnn(second_chunk)

torch.testing.assert_close(torch.cat((first_steps, second_steps), dim=1), full_steps)
torch.testing.assert_close(carried_final, full_final)
assert not torch.allclose(reset_final, full_final)

print(f"Whole phrase:     {full_final.item():+.3f}")
print(f"Chunks, carrying: {carried_final.item():+.3f}")
print(f"Chunks, reset:    {reset_final.item():+.3f}")

The whole phrase and the chunks give the same state when we carry it forward.
Starting the second chunk from zero gives a different value because the RNN
has no record of `good` at that point. The state belongs to the sequence we
are reading. In the movie-review classifier, we start a fresh state for each
review and use its final state to help predict the review's label.